In [110]:
# Importación de los datos
import pandas as pd
import numpy as np
import sqlite3
import datetime as dt

from unicodedata import normalize


## 1. CARGAR PEDIDOS (CSV)

In [111]:
# Cargar el fichero pedidos.csv
pedidos_df= pd.read_csv('data/pedidos.csv', sep="|", encoding="utf-16")
print(pedidos_df.columns.tolist())


[' Id Pedido ', ' id_cliente ', ' id_producto ', ' fecha_pedido ', ' cantidad ', ' precio_unitario ', ' total ', ' metodo_pago ', ' estado_pedido ', ' País envío ']


## 2. CARGAR CLIENTES (JSON)


In [112]:
# Cargar el fichero clientes.json
clientes_df = pd.read_json('data/clientes.json', orient='records')
print(clientes_df.columns.tolist())


['id_cliente', 'nombre', 'email', 'telefono', 'direccion', 'fecha_registro', 'genero', 'edad', 'nivel_fidelizacion']


## 3. CARGAR PRODUCTOS (SQLite)


In [113]:
# Cargar el fichero productos.db
conexion = sqlite3.connect('data/productos.db')
cursor = conexion.cursor()

productos_df = pd.read_sql_query("SELECT * FROM productos", conexion)

print(productos_df.columns.tolist())


['id_producto', 'nombre_producto', 'categoria', 'precio_base', 'descuento', 'stock', 'proveedor', 'fecha_alta']


## 4. NORMALIZAR TIPOS DE CLAVE

In [114]:
# Convertimos los valores de las columnas que actuarán de clave a str para evitar conflictos
pedidos_df.columns = pedidos_df.columns.str.strip()

pedidos_df['id_cliente'] = pedidos_df['id_cliente'].astype(str)
clientes_df['id_cliente'] = clientes_df['id_cliente'].astype(str)
productos_df['id_producto'] = productos_df['id_producto'].astype(str)

#Renombramos los campos para que sigan una nomenclatura
pedidos_df = pedidos_df.rename( columns ={
    'Id Pedido': 'id_pedido',
    'País envío': 'pais_envio'
})

## 5. UNIÓN DE DATASETS

In [148]:
# Unir los tres datasets en uno solo
clientes_pedidos = pd.merge(clientes_df, pedidos_df, on='id_cliente', how='left')
clientes_con_pedido = clientes_pedidos[clientes_pedidos['id_pedido'].notna()]
clientes_por_genero = clientes_con_pedido.groupby('genero')['id_cliente'].nunique()

print(clientes_por_genero)

pedidos_completos_df = pd.merge(productos_df, clientes_pedidos, on='id_producto', how='left')

#Cambio de tipo al dataset ID_pedido
pedidos_completos_df['id_cliente'] = pd.to_numeric(pedidos_completos_df['id_cliente'], errors='coerce')
pedidos_completos_df['id_pedido'] = pd.to_numeric(pedidos_completos_df['id_pedido'], errors='coerce')
pedidos_completos_df['id_producto'] = pd.to_numeric(pedidos_completos_df['id_producto'], errors='coerce')
#
print(f'\nColumnas del dataset completo')
print(pedidos_completos_df.columns.tolist())

print(f'\nTipos del dataset')
print(pedidos_completos_df.dtypes)

print(f'\nGeneros')
print(f'\n - Clientes')
print(clientes_df['genero'].value_counts())
print(f'\n - Pedidos Merged')
print(pedidos_completos_df['genero'].value_counts())

genero
M       4
Otro    5
Name: id_cliente, dtype: int64

Columnas del dataset completo
['id_producto', 'nombre_producto', 'categoria', 'precio_base', 'descuento', 'stock', 'proveedor', 'fecha_alta', 'id_cliente', 'nombre', 'email', 'telefono', 'direccion', 'fecha_registro', 'genero', 'edad', 'nivel_fidelizacion', 'pais', 'ciudad', 'id_pedido', 'fecha_pedido', 'cantidad', 'precio_unitario', 'total', 'metodo_pago', 'estado_pedido', 'pais_envio']

Tipos del dataset
id_producto           float64
nombre_producto        object
categoria              object
precio_base            object
descuento              object
stock                  object
proveedor              object
fecha_alta             object
id_cliente            float64
nombre                 object
email                  object
telefono               object
direccion              object
fecha_registro         object
genero                 object
edad                  float64
nivel_fidelizacion     object
pais                 

## Apartado 1: Información básica del dataset


In [116]:
# 1.1 Mostrar las primeras 5 filas del dataset global
print(f'\nMostrar las primeras 5 filas del dataset global')

print(pedidos_completos_df.head(5))

# 1.2 Mostrar las dimensiones del dataset (filas y columnas)

#pedidos_completos_df.shape

# 1.3 Mostrar los nombres de todas las columnas

print(pedidos_completos_df.columns.tolist())

# 1.4 Mostrar información sobre los tipos de datos de las columnas

print(pedidos_completos_df.dtypes)

# 1.5 Mostrar las últimas 3 filas del dataset
print(pedidos_completos_df.tail(3))



Mostrar las primeras 5 filas del dataset global
   id_producto nombre_producto categoria precio_base descuento  stock  \
0          1.0          Home X    Libros      469.99       0.0  367.0   
1          NaN      Threat Pro   Deporte      644.21      20.0   98.0   
2          3.0        Reduce X      Ropa     1417.93      15.0  309.0   
3          3.0        Reduce X      Ropa     1417.93      15.0  309.0   
4          3.0        Reduce X      Ropa     1417.93      15.0  309.0   

                         proveedor  fecha_alta  id_cliente       nombre  ...  \
0                      Allison PLC  2024-03-26         3.0  Diane Black  ...   
1                     Powers-Brown  2024-04-28         NaN          NaN  ...   
2  Patterson, Murray and Hernandez  2025-02-26         1.0  Leah Oliver  ...   
3  Patterson, Murray and Hernandez  2025-02-26         3.0  Diane Black  ...   
4  Patterson, Murray and Hernandez  2025-02-26         6.0   Ann Archer  ...   

   edad nivel_fidelizacion id_p

## Apartado 2: Tipos de datos y valores nulos


In [149]:
# 2.1 Mostrar información detallada del DataFrame (tipos, memoria, nulos)

pedidos_completos_df.info()

# 2.2 Contar valores nulos por columna

print(f'\n\nContar Valores nulos: ')
pedidos_completos_df.isnull().sum()

# 2.3 Mostrar solo las columnas que tienen valores nulos

print(pedidos_completos_df.columns[pedidos_completos_df.isnull().any()])

# 2.4 Calcular el porcentaje de valores nulos por columna

print(f'\n\nCalcular porcentaje de valores nulos')
print(((pedidos_completos_df.isnull().mean() * 100)
    .round(2)
    .loc[pedidos_completos_df.isnull().any()]))

# 2.5 Verificar si hay filas completamente vacías


if (pedidos_completos_df.isnull().all(axis=1).any()):
    print("Hay columnas nulas")
else:
    print("No hay columnas nulas")


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1218 entries, 0 to 1217
Data columns (total 27 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   id_producto         1201 non-null   float64
 1   nombre_producto     1195 non-null   object 
 2   categoria           1189 non-null   object 
 3   precio_base         1200 non-null   object 
 4   descuento           1198 non-null   object 
 5   stock               1191 non-null   object 
 6   proveedor           1196 non-null   object 
 7   fecha_alta          1195 non-null   object 
 8   id_cliente          664 non-null    float64
 9   nombre              664 non-null    object 
 10  email               664 non-null    object 
 11  telefono            664 non-null    object 
 12  direccion           664 non-null    object 
 13  fecha_registro      664 non-null    object 
 14  genero              664 non-null    object 
 15  edad                664 non-null    float64
 16  nivel_

## Bonus: Normalizacion de tabla total

In [118]:
#Formateo de valores nulos y negativos

pedidos_completos_df['total'] = pd.to_numeric(pedidos_completos_df['total'], errors='coerce') # el coerce lo que hace es que cuando haya un valor NaN no reviente y lo añade como 0
pedidos_completos_df['total'] = pedidos_completos_df['total'].fillna(0)
pedidos_completos_df.loc[pedidos_completos_df['total'] < 0, 'total'] = 0

pedidos_completos_df['cantidad'] = pd.to_numeric(pedidos_completos_df['cantidad'], errors='coerce') # el coerce lo que hace es que cuando haya un valor NaN no reviente y lo añade como 0
pedidos_completos_df['cantidad'] = pedidos_completos_df['cantidad'].fillna(0)

#########
pedidos_completos_df['precio_unitario'] = pd.to_numeric(pedidos_completos_df['precio_unitario'], errors='coerce') # el coerce lo que hace es que cuando haya un valor NaN no reviente y lo añade como 0
pedidos_completos_df['precio_unitario'] = pedidos_completos_df['precio_unitario'].fillna(0)

pedidos_completos_df['total'] = pedidos_completos_df['total'].astype(float)
pedidos_completos_df['cantidad'] = pedidos_completos_df['cantidad'].astype(float)

## Apartado 3: Estadísticas descriptivas


In [119]:
# 3.1 Estadísticas descriptivas de todas las columnas numéricas
print(f'3.1 - Estadísticas descriptivas de todas las columnas numéricas')
print(pedidos_completos_df.describe())

# 3.2 Estadísticas descriptivas de todas las columnas (numéricas y categóricas)
print(f'\n3.2 - Estadísticas descriptivas de todas las columnas (numéricas y categóricas)')
print(pedidos_completos_df.describe(include='all'))

# 3.3 Calcular la media, mediana y moda de la columna 'total'
print(f'\n3.3 - Calcular la media, mediana y moda de la columna \'total\'')
mediaProductos= pedidos_completos_df['total'].mean()
medianaProductos= pedidos_completos_df['total'].median()
modaProductos= pedidos_completos_df['total'].mode()[0]
print(f'Media: {mediaProductos}')
print(f'Mediana: {medianaProductos}')
print(f'Moda: {modaProductos}')

# 3.4 Calcular el rango (máximo - mínimo) de la columna 'cantidad'
print(f'\n3.4 - Calcular el rango (máximo - mínimo) de la columna cantidad\'')

rango = pedidos_completos_df['cantidad'].max() - pedidos_completos_df['cantidad'].min()
print(f'\nEl rango máximo de la columna cantidad es de: {rango}')
# 3.5 Calcular la desviación estándar y la varianza de la columna 'total'
print(f'\n3.5 - Calcular la desviación estándar y la varianza de la columna \'total\'')
desviacion= pedidos_completos_df['total'].std()
varianza = pedidos_completos_df['total'].var()
print(f'Desviacion: {desviacion}')
print(f'Varianza: {varianza}')

# 3.6 Contar valores únicos en columnas categóricas
print(f'\n\n3.6 - Contar valores únicos en columnas categóricas')
pedidos_completos_df.select_dtypes(include='object').apply(lambda col: col.astype(str).nunique())


3.1 - Estadísticas descriptivas de todas las columnas numéricas
        id_producto  id_cliente        edad     id_pedido      cantidad  \
count   1201.000000  664.000000  664.000000    642.000000   1218.000000   
mean   -7209.551207    2.736446   51.128012  18727.001558   -571.763547   
std    26738.184102    1.657233   22.286633  16297.011088   7562.418675   
min   -99999.000000    1.000000   29.000000 -99999.000000 -99999.000000   
25%      167.000000    1.000000   32.000000   9541.500000      0.000000   
50%      442.000000    2.500000   35.000000  19286.500000      1.000000   
75%      725.000000    4.000000   75.000000  29288.250000      6.000000   
max     1000.000000    9.000000   87.000000  39968.000000     10.000000   

       precio_unitario         total  
count      1218.000000   1218.000000  
mean         28.287923   3014.474278  
std        7070.737539   4460.515561  
min      -99999.000000      0.000000  
25%           0.000000      0.000000  
50%         118.205000    

nombre_producto       858
categoria              14
precio_base           956
descuento              12
stock                 426
proveedor             945
fecha_alta            637
nombre                 10
email                  10
telefono               10
direccion              10
fecha_registro         10
genero                  3
nivel_fidelizacion      5
fecha_pedido          384
metodo_pago            10
estado_pedido          11
pais_envio            223
dtype: int64

## Apartado 4: Filtrado simple

In [120]:
# 4.1 Filtrar pedidos con cantidad mayor a 5 unidades
print('4.1 - Filtrar pedidos con cantidad mayor a 5 unidades')

print((pedidos_completos_df['id_pedido'][pedidos_completos_df['cantidad'] >5].head(4)))
print("\n")

# 4.2 Filtrar pedidos con total mayor a 1000 euros
print('\n4.2 - Filtrar pedidos con total mayor a 1000 euros')

print(pedidos_completos_df['id_pedido'][pedidos_completos_df['total'] >1000].head(4))

# 4.3 Filtrar pedidos con precio unitario menor a 50 euros
print(f'\n4.3 - Filtrar pedidos con precio unitario menor a 50 euros')

print(
    pedidos_completos_df[
        (pedidos_completos_df['precio_unitario'] < 50) &
        (pedidos_completos_df['id_pedido'].notna())
    ]['id_pedido'].head(4)
)

# 4.4 Filtrar pedidos con cantidad igual a 1
print(f'\n4.4 - Filtrar pedidos con cantidad igual a 1')

print(
    pedidos_completos_df[
        pedidos_completos_df['cantidad'] == 1
    ][['id_pedido', 'nombre_producto', 'cantidad']].head(4)
)


# 4.5 Mostrar los primeros 5 pedidos grandes (cantidad > 5)

print(f'\n4.5 - Mostrar los primeros 5 pedidos grandes (cantidad > 5)')
print(
    pedidos_completos_df[
        pedidos_completos_df['cantidad'] >5
        ][['id_pedido', 'nombre_producto', 'cantidad']].sort_values('cantidad', ascending=False).head(5)
)

# 4.6 Mostrar los primeros 5 pedidos caros (total > 1000€)

print(f'\n4.6 - Mostrar los primeros 5 pedidos caros (total > 1000€)')

filtro= pedidos_completos_df[
        pedidos_completos_df['cantidad'] * pedidos_completos_df['precio_unitario'] > 1000
    ].copy()

filtro['total'] = filtro['cantidad'] * filtro['precio_unitario']

print(
    filtro[['id_pedido', 'nombre_producto', 'cantidad', 'precio_unitario','total']].head(5)
)
#Necesite el filtro para poder evitar el error de CopyWarning y para poder ver la cantidad total de cantidad * precio


4.1 - Filtrar pedidos con cantidad mayor a 5 unidades
2      4616.0
5     26654.0
7     10194.0
18     3302.0
Name: id_pedido, dtype: float64



4.2 - Filtrar pedidos con total mayor a 1000 euros
0    28171.0
2     4616.0
3     7941.0
4    22670.0
Name: id_pedido, dtype: float64

4.3 - Filtrar pedidos con precio unitario menor a 50 euros
3       7941.0
78      9539.0
86     10919.0
105    29540.0
Name: id_pedido, dtype: float64

4.4 - Filtrar pedidos con cantidad igual a 1
    id_pedido nombre_producto  cantidad
27     3711.0       Pull Lite       1.0
66    28137.0           Top X       1.0
67     5892.0  Challenge Lite       1.0
85    34264.0        Form Pro       1.0

4.5 - Mostrar los primeros 5 pedidos grandes (cantidad > 5)
      id_pedido nombre_producto  cantidad
1214    23300.0      Degree Pro      10.0
1154    38365.0          Room X      10.0
1118     5521.0       Drug Lite      10.0
1132    21427.0       Still Max      10.0
589     29504.0       Would Pro      10.0

4.6 - Mo

## Apartado 5: Filtrado con múltiples condiciones

In [121]:
# 5.1 Filtrar pedidos enviados con cantidad mayor a 3
print(f'5.1 Filtrar pedidos enviados con cantidad mayor a 3')

print(
    pedidos_completos_df[
        (pedidos_completos_df['estado_pedido']=="Enviado")
        & (pedidos_completos_df['cantidad'] > 3)
    ][['id_pedido', 'nombre_producto', 'estado_pedido', 'cantidad']].head(4)
)

# 5.2 Filtrar pedidos de tarjeta o PayPal con total mayor a 500 euros
print(f'\n5.2 - Filtrar pedidos de tarjeta o PayPal con total mayor a 500 euros')

print(
    pedidos_completos_df[
        (pedidos_completos_df['metodo_pago'] == "PayPal") |
        (pedidos_completos_df['metodo_pago'] == "Tarjeta" ) &
        (pedidos_completos_df['total'] > 500)
    ][['id_pedido', 'nombre_producto', 'metodo_pago', 'total']].head(4)
)

# 5.3 Filtrar pedidos pendientes o cancelados con cantidad menor a 2
print(f'\n5.3 - Filtrar pedidos pendientes o cancelados con cantidad menor a 2')

print(
    pedidos_completos_df[
        (pedidos_completos_df['estado_pedido'] == "Cancelado") |
        (pedidos_completos_df['estado_pedido'] == "Pendientes") &
        (pedidos_completos_df['cantidad'] > 2)
    ][['id_pedido','nombre_producto', 'estado_pedido', 'cantidad' ]].head(4)
)


# 5.4 Filtrar pedidos enviados o devueltos con precio unitario mayor a 100
print(f'\n5.4 - Filtrar pedidos enviados o devueltos con precio unitario mayor a 100')

print(
    pedidos_completos_df[
        (pedidos_completos_df['estado_pedido'] == "Enviado") |
        (pedidos_completos_df['estado_pedido'] == "Cancelado") &
        (pedidos_completos_df['cantidad'] > 100)
    ][['id_pedido', 'nombre_producto', 'estado_pedido','cantidad']].head(4)
)
# 5.5 Mostrar los primeros 5 pedidos enviados grandes (estado - pedido = enviado) (cantidad > 3)
print(f'\n5.5 - Mostrar los primeros 5 pedidos enviados grandes (estado - pedido = enviado) (cantidad > 3)')

print(
    pedidos_completos_df[
        (pedidos_completos_df['estado_pedido'] == "Enviado") &
        (pedidos_completos_df['cantidad'] > 3)
    ][['id_pedido', 'estado_pedido', 'cantidad']].head(4)
)
# 5.6 Mostrar los primeros 5 pedidos premium (metodo pago tarjeta, paypal) (total > 500)
print(f'\n5.6 - Mostrar los primeros 5 pedidos premium (metodo pago tarjeta, paypal) (total > 500)')

print(
    pedidos_completos_df[
        (pedidos_completos_df['metodo_pago'] == "Tarjeta") |
        (pedidos_completos_df['metodo_pago'] == "PayPal") &
        (pedidos_completos_df['total'] > 500)
    ][['id_pedido', 'metodo_pago', 'total']].sort_values('total', ascending=False).head(5)
)

5.1 Filtrar pedidos enviados con cantidad mayor a 3
    id_pedido nombre_producto estado_pedido  cantidad
5     26654.0        Half Max       Enviado       6.0
12     9217.0          Very X       Enviado       4.0
51     3593.0      Great Lite       Enviado       6.0
62    11589.0          Blue X       Enviado       5.0

5.2 - Filtrar pedidos de tarjeta o PayPal con total mayor a 500 euros
   id_pedido nombre_producto metodo_pago    total
0    28171.0          Home X     Tarjeta  6969.45
3     7941.0        Reduce X      PayPal  3410.66
6     4950.0     Control Pro      PayPal  3671.55
8    35871.0  Everything Max      PayPal  2541.24

5.3 - Filtrar pedidos pendientes o cancelados con cantidad menor a 2
    id_pedido     nombre_producto estado_pedido  cantidad
7     10194.0         Control Pro     Cancelado       7.0
18     3302.0            High Pro     Cancelado       7.0
21    18886.0  Responsibility Pro     Cancelado       2.0
24     9093.0  Responsibility Pro     Cancelado       4

## Apartado 6: Filtrado temporal

In [122]:
# 6.1 Convertir la columna fecha_pedido a tipo datetime
#Como hay fechas que son 0X, python no lo permite desde la version 3. Solucion:
pedidos_completos_df['fecha_pedido'] = pedidos_completos_df['fecha_pedido'].astype(str)

print(f'\n6.1 - Convertir la columna fecha_pedido a tipo datetime')
pedidos_completos_df['fecha_pedido'] = pd.to_datetime(
    pedidos_completos_df['fecha_pedido'], format="%Y-%m-%d", errors='coerce'
)

print(
    pedidos_completos_df[['id_pedido', 'fecha_pedido']].dtypes
)

# 6.2 Filtrar pedidos del año 2024
print(f'\n6.2 - Filtrar pedidos del año 2024')

print(
    pedidos_completos_df.loc[
        pedidos_completos_df['fecha_pedido'].dt.year == 2024,
        ['id_pedido', 'fecha_pedido', 'nombre_producto']
    ].head(4)
)

# 6.3 Filtrar pedidos del mes de enero de 2024
print(f'\n6.3 - Filtrar pedidos del mes de enero de 2024')

print(pedidos_completos_df.loc[
    pedidos_completos_df['fecha_pedido'].dt.month == 1,
    ['id_pedido', 'fecha_pedido', 'nombre_producto']
      ].head(4))

# 6.4 Filtrar pedidos del primer trimestre de 2024
print(f'\n6.4 - Filtrar pedidos del primer trimestre de 2024')

#Hay dos formas between y dos condiciones
primer_trimestre= pedidos_completos_df.loc[
    (pedidos_completos_df['fecha_pedido'].dt.month <= 3) &
    (pedidos_completos_df['fecha_pedido'].dt.year == 2024)
    ][['id_pedido','fecha_pedido', 'nombre_producto']].head(4)
print(primer_trimestre)

print("#"*4, " Segundo Método ", "#"*4)

primer_trimestre_between= pedidos_completos_df.loc[
    (pedidos_completos_df['fecha_pedido'].between('2024-01-01','2024-03-31'))
][['id_pedido','fecha_pedido', 'nombre_producto']].head(4)

print(primer_trimestre_between)

# 6.5 Filtrar pedidos de los lunes (día de la semana = 0)
print(f'\n6.5 - Filtrar pedidos de los lunes (día de la semana = 0)')

print(pedidos_completos_df.loc[
          (pedidos_completos_df['fecha_pedido'].dt.weekday == 0)
      ][['id_pedido', 'fecha_pedido', 'nombre_producto']].head(4)
      )

# 6.6 Mostrar los primeros 5 pedidos de 2024
print(f'\n6.6 - Mostrar los primeros 5 pedidos de 2024')
print(pedidos_completos_df.loc[
          (pedidos_completos_df['fecha_pedido'].dt.year == 2024)
      ][['id_pedido','nombre_producto', 'fecha_pedido']]
      .sort_values('fecha_pedido', ascending=True).head(5)
      )


6.1 - Convertir la columna fecha_pedido a tipo datetime
id_pedido              float64
fecha_pedido    datetime64[ns]
dtype: object

6.2 - Filtrar pedidos del año 2024
   id_pedido fecha_pedido nombre_producto
2     4616.0   2024-05-17        Reduce X
3     7941.0   2024-05-10        Reduce X
4    22670.0   2024-04-28        Reduce X
5    26654.0   2024-06-09        Half Max

6.3 - Filtrar pedidos del mes de enero de 2024
     id_pedido fecha_pedido    nombre_producto
41         NaN   2024-01-14           Help Pro
53     19582.0   2025-01-07             At Max
113    25133.0   2024-01-24  Organization Lite
134    32508.0   2024-01-12       Discussion X

6.4 - Filtrar pedidos del primer trimestre de 2024
     id_pedido fecha_pedido nombre_producto
41         NaN   2024-01-14        Help Pro
71     26292.0   2024-03-14    Increase Pro
76      3042.0   2024-03-17      Crime Lite
107    34274.0   2024-03-22       Able Lite
####  Segundo Método  ####
     id_pedido fecha_pedido nombre_prod

## Apartado 7: Estado del pedido

In [123]:
# 7.1 Contar el número de pedidos por cada estado
print(f'7.1 - Contar el número de pedidos por cada estado')

print(pedidos_completos_df['estado_pedido'].value_counts())
# 7.2 Calcular el porcentaje de pedidos por estado
print(f'\n7.2 - Calcular el porcentaje de pedidos por estado')
porcentaje = pedidos_completos_df['estado_pedido'].value_counts(normalize=True) * 100
print(porcentaje.round(2))

# 7.3 Calcular el total de ventas por estado
print(f'\n7.3 - Calcular el total de ventas por estado')

## Como los valores de estas columnas estan truncados, necesito normalizarlo para que no haya strings mal puestas
## o numeros negativos que corrompan el resultado
pedidos_completos_df['cantidad'] = pedidos_completos_df['cantidad'].replace(-99999.0, 0)
pedidos_completos_df['precio_unitario'] = pedidos_completos_df['precio_unitario'].replace(-99999.0, 0)


pedidos_completos_df['total_estado'] = pedidos_completos_df['cantidad'] * pedidos_completos_df['precio_unitario']
total_ventas_estado = pedidos_completos_df.groupby('estado_pedido')['total_estado'].sum()
print(total_ventas_estado)

# 7.4 Calcular el valor promedio de pedidos por estado
print(f'\n7.4 - Calcular el valor promedio de pedidos por estado')
media_ventas_estado = pedidos_completos_df.groupby('estado_pedido')['cantidad'].mean()
print(round(media_ventas_estado,2))

# 7.5 Calcular la cantidad total de productos vendidos por estado
print(f'\n7.5 - Calcular la cantidad total de productos vendidos por estado')

total_enviado = pedidos_completos_df[pedidos_completos_df['estado_pedido'] == "Enviado"]['cantidad'].sum()
print(total_enviado)

# 7.6 Mostrar estadísticas completas por estado
print(f'\n7.6 - Mostrar estadísticas completas por estado')
estadisticas_estado = pedidos_completos_df.groupby('estado_pedido').describe()
print(estadisticas_estado)


7.1 - Contar el número de pedidos por cada estado
estado_pedido
Devuelto       176
Cancelado      166
Pendiente      161
Enviado        137
###ERROR###      7
De               6
Pe               3
Ca               2
##               1
En               1
Name: count, dtype: int64

7.2 - Calcular el porcentaje de pedidos por estado
estado_pedido
Devuelto       26.67
Cancelado      25.15
Pendiente      24.39
Enviado        20.76
###ERROR###     1.06
De              0.91
Pe              0.45
Ca              0.30
##              0.15
En              0.15
Name: proportion, dtype: float64

7.3 - Calcular el total de ventas por estado
estado_pedido
##                827.43
###ERROR###     41104.68
Ca              10959.38
Cancelado      798414.50
De              15701.34
Devuelto       969816.51
En                589.12
Enviado        735951.75
Pe              18063.16
Pendiente      904254.67
Name: total_estado, dtype: float64

7.4 - Calcular el valor promedio de pedidos por estado
estado_ped

## Bonus normalizacion Método de pago


In [124]:
map_metodo_pago = {
    "Cr": "Criptomoneda",
    "Pa": "PayPal",
    "Ta": "Tarjeta",
    "Tr": "Transferencia"
}
## Como estan acortados, voy a crear un map con las correspondencias y asignárselo al dataframe para corregir esta situación
pedidos_completos_df['metodo_pago'] = (
    pedidos_completos_df['metodo_pago'].replace(map_metodo_pago)
)

## Apartado 8: Método de pago

In [125]:
# 8.1 Contar el número de pedidos por método de pago
print(f'\n8.1 -Contar el número de pedidos por método de pago')

metodos_pago = pedidos_completos_df.groupby('metodo_pago').size()
print(metodos_pago)

# 8.2 Calcular el porcentaje de uso de cada método de pago
print(f'\n8.2 - Calcular el porcentaje de uso de cada método de pago')

porcentaje = pedidos_completos_df['metodo_pago'].value_counts(normalize=True) * 100
print(porcentaje.round(2))

# 8.3 Calcular el valor total de ventas por método de pago
print(f'\n8.3 - Calcular el valor total de ventas por método de pago')

ventas_por_metodo = pedidos_completos_df.groupby('metodo_pago')['total'].sum()
print(ventas_por_metodo)

# 8.4 Calcular el valor promedio de pedidos por método de pago
print(f'\n8.4 - Calcular el valor promedio de pedidos por método de pago')

valor_promedio =(
    pedidos_completos_df.groupby('metodo_pago')['total']
    .mean()
    .round(2)
    .sort_values(ascending=False)
)
print(valor_promedio)

# 8.5 Calcular la cantidad promedio de productos por método de pago
print(f'\n8.5 - Calcular la cantidad promedio de productos por método de pago')
valor_promedio_productos = (
    pedidos_completos_df.groupby('metodo_pago')['cantidad']
    .mean()
    .round(2)
    .sort_values(ascending=False)
)

print(valor_promedio_productos)

# 8.6 Mostrar estadísticas completas por método de pago (numero de pedidos, total de ventas, promedio de pedidos, cantidad total, promedio de la cantidad)
print(f'\n8.6 - Mostrar estadísticas completas por método de pago (numero de pedidos, total de ventas, promedio de pedidos, cantidad total, promedio de la cantidad)')
estadisticas_pago = pedidos_completos_df.groupby('metodo_pago').agg(
    numero_pedidos=('id_pedido', 'count'),
    total_ventas=('total', 'sum'),
    promedio_pedido=('total', 'mean'),
    cantidad_total=('cantidad', 'sum'),
    promedio_cantidad=('cantidad', 'mean')
).round(2)

# Ordenar por número de pedidos de mayor a menor (opcional)
estadisticas_pago = estadisticas_pago.sort_values(by='numero_pedidos', ascending=False)

print(estadisticas_pago)



8.1 -Contar el número de pedidos por método de pago
metodo_pago
###ERROR###       10
Criptomoneda     174
PayPal           157
Tarjeta          166
Transferencia    153
dtype: int64

8.2 - Calcular el porcentaje de uso de cada método de pago
metodo_pago
Criptomoneda     26.36
Tarjeta          25.15
PayPal           23.79
Transferencia    23.18
###ERROR###       1.52
Name: proportion, dtype: float64

8.3 - Calcular el valor total de ventas por método de pago
metodo_pago
###ERROR###        48645.06
Criptomoneda     1072407.54
PayPal            849009.38
Tarjeta           945335.61
Transferencia     747105.63
Name: total, dtype: float64

8.4 - Calcular el valor promedio de pedidos por método de pago
metodo_pago
Criptomoneda     6163.26
Tarjeta          5694.79
PayPal           5407.70
Transferencia    4883.04
###ERROR###      4864.51
Name: total, dtype: float64

8.5 - Calcular la cantidad promedio de productos por método de pago
metodo_pago
Criptomoneda     5.90
###ERROR###      5.50
Tar

## Apartado 9: País de envío

In [126]:
# 9.1 Contar el número total de países únicos
print(f'9.1 - Contar el número total de países únicos')

paises_unicos = pedidos_completos_df['pais_envio'].nunique()
print(f'\nHay un numero de {paises_unicos} paises únicos en el dataframe')

# 9.2 Mostrar los top 10 países por número de pedidos
print(f'\n9.2 - Mostrar los top 10 países por número de pedidos')

paises_10_vendidos = (
    pedidos_completos_df['pais_envio']
    .value_counts()
    .head(10)
)

print(paises_10_vendidos)


# 9.3 Mostrar los top 10 países por valor total de ventas
print("\n9.3 - Top 10 países por valor total de ventas")

top_paises_ventas = (
    pedidos_completos_df.groupby('pais_envio')['total']
    .sum()
    .sort_values(ascending=False)
    .head(10)
)

print(top_paises_ventas)

# 9.4 Calcular el valor promedio de pedidos por país (top 10)
print('\n9.4 - Calcular el valor promedio de pedidos por país (top 10)')

valor_medio = (
    pedidos_completos_df.groupby('pais_envio')['total']
    .mean()
    .round(2)
    .sort_values(ascending=False)
    .head(10)
)

print(valor_medio)

# 9.5 Calcular la cantidad total de productos enviados por país (top 10)
print('\n9.5 - Calcular la cantidad total de productos enviados por país (top 10)')

cantidad_enviada = (
    pedidos_completos_df[pedidos_completos_df['estado_pedido'] == "Enviado"]
    .groupby('pais_envio')['cantidad']          #La condición se debe poner antes que el groupby ya que no acepta booleanos
    .sum()
    .sort_values(ascending=False)
    .head(10)
)

print(cantidad_enviada)


# 9.6 Mostrar estadísticas del país con más pedidos (número de pedidos, total de ventas, promedio de pedido y total de cantidad)
print(f'\n9.6 - Mostrar estadísticas del país con más pedidos (número de pedidos, total de ventas, promedio de pedido y total de cantidad)')
pais_top =(
    pedidos_completos_df['pais_envio']
    .value_counts()
    .idxmax()
)

#Recojo los pedidos del pais en un dataframe.
df_pais_top = pedidos_completos_df[pedidos_completos_df['pais_envio'] == pais_top]

# Calculo estadisticas
num_pedidos = len(df_pais_top)
total_ventas = df_pais_top['total'].sum()
promedio_pedido = df_pais_top['total'].mean()
total_cantidad = df_pais_top['cantidad'].sum()

print("\nEstadísticas:")
print(f"- Número de pedidos: {num_pedidos}")
print(f"- Total de ventas: {total_ventas:.2f}")
print(f"- Promedio de pedido: {promedio_pedido:.2f}")
print(f"- Cantidad total enviada: {total_cantidad}")

9.1 - Contar el número total de países únicos

Hay un numero de 222 paises únicos en el dataframe

9.2 - Mostrar los top 10 países por número de pedidos
pais_envio
Mauritania           16
Macao                15
Christmas Island     15
Tonga                15
Equatorial Guinea    15
Cyprus               14
Cote d'Ivoire        14
Cambodia              8
###ERROR###           8
Switzerland           7
Name: count, dtype: int64

9.3 - Top 10 países por valor total de ventas
pais_envio
Cyprus               205646.35
Cote d'Ivoire        174422.20
Christmas Island     125123.60
Equatorial Guinea    117309.54
Mauritania           106776.93
Switzerland           65491.46
Slovenia              57324.60
Argentina             49110.94
Cambodia              42918.77
Egypt                 42179.18
Name: total, dtype: float64

9.4 - Calcular el valor promedio de pedidos por país (top 10)
pais_envio
Antarctica (the territory South of 60 deg S)    19437.50
Iceland                                    

## Apartado 10: Análisis demográfico

In [145]:
# 10.1 Contar el número total de clientes únicos
print(f'10.1 - Contar el número total de clientes únicos')

num_clientes = clientes_df['id_cliente'].nunique()

print(f'\nHay {num_clientes} clientes en el dataframe.')

# 10.2 Distribución de clientes por género
print(f'\n10.2 - Distribucion de clientes por género.')

distribucion_genero = clientes_df.groupby('genero').size()
print(distribucion_genero)

# 10.3 Porcentaje de distribución por género
print("\n10.3 - Porcentaje de distribución por género")

porcentaje_genero = clientes_df['genero'].value_counts(normalize=True) * 100
porcentaje_genero = porcentaje_genero.round(2)

print(porcentaje_genero)

# 10.4 Distribución de clientes por nivel de fidelización
print(f'\n10.4 - Distribución de clientes por nivel de fidelización')

distribucion_fidelizacion = clientes_df['nivel_fidelizacion'].value_counts()
print(distribucion_fidelizacion)

# 10.5 Estadísticas básicas de edad
print("\n10.5 - Estadísticas básicas de edad")

estadisticas_edad = clientes_df['edad'].describe().round(2)
print(estadisticas_edad)

# 10.6 Mostrar estadísticas completas de edad por género (número de clientes, edad promedio, edad mínima, edad máxima y mediana de edad)
estadisticas_genero = (
    pedidos_completos_df.groupby('genero')
    .agg(
        num_clientes=('id_cliente', 'nunique'),
        edad_promedio=('edad', 'mean'),
        edad_min=('edad', 'min'),
        edad_max=('edad', 'max'),
        mediana_edad=('edad', 'median')
    )

)
print(f'\npedidos COmpletos')

print(pedidos_completos_df['genero'].value_counts())
print(f'\nclientes')
print(clientes_df['genero'].value_counts())

print(estadisticas_genero)




10.1 - Contar el número total de clientes únicos

Hay 5000 clientes en el dataframe.

10.2 - Distribucion de clientes por género.
genero
F       1677
M       1634
Otro    1689
dtype: int64

10.3 - Porcentaje de distribución por género
genero
Otro    33.78
F       33.54
M       32.68
Name: proportion, dtype: float64

10.4 - Distribución de clientes por nivel de fidelización
nivel_fidelizacion
Oro        1258
Bronce     1254
Platino    1248
Plata      1240
Name: count, dtype: int64

10.5 - Estadísticas básicas de edad
count    5000.00
mean       54.16
std        21.05
min        18.00
25%        36.00
50%        54.00
75%        73.00
max        90.00
Name: edad, dtype: float64

pedidos COmpletos
genero
Otro    498
M       166
Name: count, dtype: int64

clientes
genero
Otro    1689
F       1677
M       1634
Name: count, dtype: int64
        num_clientes  edad_promedio  edad_min  edad_max  mediana_edad
genero                                                               
M                

## Apartado 11: Clientes por país

In [128]:
# 11.1 Contar el número total de países únicos donde viven los clientes
print("\n11.1 - Número total de países únicos donde viven los clientes")

#extrae el pais de direccion
clientes_df.loc[:, 'pais'] = clientes_df['direccion'].apply(
    lambda x: x['pais'].strip() if isinstance(x, dict) and 'pais' in x else None
)

# Contar países únicos
paises_unicos = clientes_df['pais'].nunique()
print(f'\nHay clientes por todos los {paises_unicos} paises del dataframe')

# 11.2 Mostrar los top 15 países con más clientes
print(f'\n11.2 - Mostrar los top 15 países con más clientes')
top_paises_clientes = (
    clientes_df.groupby('pais')['id_cliente']
    .nunique()
    .sort_values(ascending=False)
    .head(15)
)

print(top_paises_clientes)

# 11.3 Calcular el porcentaje de clientes por país (top 10)
print(f'\nCalcular el porcentaje de clientes por país (top 10)')

top10_paises_clientes = (
    pedidos_completos_df['pais_envio']
    .value_counts(normalize=True)
    .head(10) * 100
)
print(top10_paises_clientes)

# 11.4 Mostrar estadísticas (número de clientes, edad promedio, edad mínima, edad máxima y mediana de edad) de edad por país (top 5 países)
print(f'\nMostrar estadísticas (número de clientes, edad promedio, edad mínima, edad máxima y mediana de edad) de edad por país (top 5 países)')
top5_paises_clientes = (
    clientes_df.groupby('pais')['id_cliente']
    .nunique()
    .sort_values(ascending=False)
    .head(5)
    .index
)

#Saco los indices de los top 5 paises y cojo el dataframe con ellos dentro
#Si el pais cumple con la condicion de arriba, lo añade al dataframe de respuesta
top5_df = pedidos_completos_df[pedidos_completos_df['pais_envio'].isin(top5_paises_clientes)]

estadisticas = top5_df.groupby('pais_envio')['edad'].agg(
    ['count', 'mean', 'min', 'max', 'median']
)
print(estadisticas)


# 11.5 Mostrar distribución de género por país (top 3 países)


# 11.6 Mostrar distribución de nivel de fidelización por país (top 3 países)





11.1 - Número total de países únicos donde viven los clientes

Hay clientes por todos los 243 paises del dataframe

11.2 - Mostrar los top 15 países con más clientes
pais
Korea                  50
Congo                  38
Kenya                  34
Honduras               31
Vanuatu                31
Cayman Islands         30
Denmark                30
Mauritius              30
Comoros                30
Antigua and Barbuda    29
San Marino             29
Poland                 28
Bermuda                28
Togo                   28
Papua New Guinea       28
Name: id_cliente, dtype: int64

Calcular el porcentaje de clientes por país (top 10)
pais_envio
Mauritania           2.453988
Macao                2.300613
Christmas Island     2.300613
Tonga                2.300613
Equatorial Guinea    2.300613
Cyprus               2.147239
Cote d'Ivoire        2.147239
Cambodia             1.226994
###ERROR###          1.226994
Switzerland          1.073620
Name: proportion, dtype: float64

Mostrar 

## Apartado 12: Clientes por ciudad

In [129]:
# 12.1 Mostrar las top 20 ciudades con más clientes
print(f'\nMostrar las top 20 ciudades con más clientes')

# En este caso, para tener todos los clientes de antes del merge, es mejor usar la tabla clientes_df
# Extraer la ciudad desde el diccionario 'direccion'

clientes_df.loc[:, 'ciudad'] = clientes_df['direccion'].apply(
    lambda x: x['ciudad'].strip() if isinstance(x, dict) and 'ciudad' in x else None
)

# Contar clientes por ciudad y que sean unicos, por si acaso hubiese repetidos
top_ciudades = (
    clientes_df.groupby('ciudad')['id_cliente']
    .nunique()
    .sort_values(ascending=False)
    .head(20)
)
print(top_ciudades)

# 12.2 Mostrar las ciudades con exactamente 1 cliente
print(f'\nMostrar las ciudades con exactamente 1 cliente')
ciudades_con_1_cliente = (
    clientes_df.groupby('ciudad')['id_cliente']
    .nunique()
    .loc[lambda x: x == 1]          # filtrar solo las que tienen exactamente 1
)

print(ciudades_con_1_cliente)
# 12.3 Mostrar las ciudades con más de 10 clientes
print(f'\nMostrar las ciudades con más de 10 clientes')
ciudades_con_1_cliente = (
    clientes_df.groupby('ciudad')['id_cliente']
    .nunique()
    .loc[lambda x: x >= 10]
    .sort_values(ascending=False) # filtrar solo las que tienen exactamente 1
)

print(ciudades_con_1_cliente)


Mostrar las top 20 ciudades con más clientes
ciudad
New Michael        8
West Jessica       7
South Michael      6
Port Matthew       6
Williamsmouth      6
Lake John          5
New Christopher    5
New Jessica        5
Port David         5
East James         5
Lake Joseph        5
Lake Michael       5
South Jennifer     5
New Danielle       5
East Nicole        4
East Matthew       4
New Emily          4
North Matthew      4
South Kimberly     4
North Beth         4
Name: id_cliente, dtype: int64

Mostrar las ciudades con exactamente 1 cliente
ciudad
Aaronborough     1
Aaronport        1
Aaronview        1
Adambury         1
Adamland         1
                ..
Yvonneborough    1
Zacharyland      1
Zacharyside      1
Zavalaland       1
Zimmermanview    1
Name: id_cliente, Length: 3765, dtype: int64

Mostrar las ciudades con más de 10 clientes
Series([], Name: id_cliente, dtype: int64)


## Apartado 13: Categoría de productos

In [130]:
# 13.1 Mostrar las categorías cuyo nombre tiene más de 6 caracteres y que tienen entre 50 y 160 productos
print(f'\nMostrar las categorías cuyo nombre tiene más de 6 caracteres y que tienen entre 50 y 160 productos')

categorias = pedidos_completos_df.groupby('categoria')['id_pedido'].count()
categorias_final = categorias[
    (categorias.between(50, 160)) &
    (categorias.index.str.len() > 6)
]
print(categorias_final)
# 13.2 Mostrar las categorías que contienen la letra "o" (mayúscula o minúscula) y tienen menos de 120 productos
print(f'\nMostrar las categorías que contienen la letra "o" (mayúscula o minúscula) y tienen menos de 120 productos')

categorias = pedidos_completos_df.groupby('categoria')['id_pedido'].count()

#Con index accedo al nombre de la columna de la que agrupe anteriormente, y con str accedo a funcione de las strings
contienen_o = categorias[categorias.index.str.contains('o', case=False)]    # Como tiene que ser Case Insensitive, se usa el False para no distinguir mayus de minus
categorias_final = contienen_o[contienen_o < 120].index

print(categorias_final)



Mostrar las categorías cuyo nombre tiene más de 6 caracteres y que tienen entre 50 y 160 productos
categoria
Deporte        109
Electrónica     92
Juguetes       100
Name: id_pedido, dtype: int64

Mostrar las categorías que contienen la letra "o" (mayúscula o minúscula) y tienen menos de 120 productos
Index(['###ERROR###', 'Deporte', 'Ho', 'Hogar', 'Libros', 'Ro', 'Ropa'], dtype='object', name='categoria')


## Bonus normalizacion Categorias

In [131]:
map_categoria = {
    "El": "Electrónica",
    "Ho": "Hogar",
    "Ju": "Juguetes",
    "Ro": "Ropa",
    "Li": "Libros",
    "De": "Deporte"
}
## Como estan acortados, voy a crear un map con las correspondencias y asignárselo al dataframe para corregir esta situación
pedidos_completos_df['categoria'] = (
    pedidos_completos_df['categoria'].replace(map_categoria)
)

pedidos_completos_df['categoria'].unique()



array(['Libros', 'Deporte', 'Ropa', 'Hogar', 'Juguetes', '###ERROR###',
       'Electrónica', None], dtype=object)

## Apartado 14: Precios

In [132]:
# 14.1 Mostrar estadísticas básicas de precios base (precio promedio, precio mínimo, precio máximo y mediana)


# 14.2 Calcular el precio promedio por categoría
print(f'\nCalcular el precio promedio por categoría')
precio_promedio= pedidos_completos_df.groupby('categoria')['precio_unitario'].mean()
print(precio_promedio)


# 14.3 Encontrar los productos más caros y más baratos
print(f'\nEncontrar los productos más caros y más baratos')
precio_min = pedidos_completos_df['precio_unitario'].min()
precio_max = pedidos_completos_df['precio_unitario'].max()

# Filtrar los productos con ese precio
producto_mas_barato = pedidos_completos_df[pedidos_completos_df['precio_unitario'] == precio_min][['id_producto', 'cantidad', 'precio_unitario']].head(3)
producto_mas_caro = pedidos_completos_df[pedidos_completos_df['precio_unitario'] == precio_max][['id_producto', 'cantidad', 'precio_unitario']]
print(f'\nProductos mas baratos:\n')
print(producto_mas_barato)

print(f'\nProductos mas caros:\n')
print(producto_mas_caro)


# 14.4 Mostrar los top 10 productos más caros

print(f'\n Mostrar los top 10 productos más caros')
top_10_caros = pedidos_completos_df.nlargest(10, 'precio_unitario')
print(top_10_caros)


Calcular el precio promedio por categoría
categoria
###ERROR###    508.388261
Deporte        545.091970
Electrónica    500.384619
Hogar          526.755932
Juguetes       510.450053
Libros         530.014238
Ropa           490.957259
Name: precio_unitario, dtype: float64

Encontrar los productos más caros y más baratos

Productos mas baratos:

   id_producto  cantidad  precio_unitario
1          NaN       0.0              0.0
3          3.0       2.0              0.0
9          7.0       0.0              0.0

Productos mas caros:

     id_producto  cantidad  precio_unitario
543        443.0       6.0           1991.8

 Mostrar los top 10 productos más caros
      id_producto nombre_producto    categoria precio_base descuento   stock  \
543         443.0     Street Lite  Electrónica       78.05       0.0   470.0   
564         458.0    Million Lite       Libros      866.49       0.0   426.0   
858         698.0      Start Lite     Juguetes     1619.15      15.0   213.0   
17           

## Apartado 15: Productos más vendidos

In [133]:
# 15.1 Obtener el producto más vendido (cantidad vendida, total vendido, nombre del producto y categoría)
resumen_productos = pedidos_completos_df.groupby(
    ['nombre_producto', 'categoria'], as_index=False
).agg({'cantidad': 'sum', 'total': 'sum'})

# Obtener el producto más vendido en cantidad
producto_mas_vendido = resumen_productos.sort_values(by='cantidad', ascending=False).iloc[0]

print("Producto más vendido:")
print(producto_mas_vendido)



Producto más vendido:
nombre_producto    ###ERROR###
categoria               Libros
cantidad                  45.0
total                 52992.37
Name: 4, dtype: object


## Apartado 16: Análisis temporal


In [134]:
# 16.1 Obtener información sobre las ventas por mes (número total de ventas, número de pedidos y cantidad total)
print("\nInformación sobre las ventas por mes")

# Crea una columna 'mes' en formato YYYY-MM para diferenciar mes y año
pedidos_completos_df['mes'] = pedidos_completos_df['fecha_pedido'].dt.to_period('M').astype(str)

ventas_por_mes = (
    pedidos_completos_df.groupby('mes')
    .agg({
        'total': 'sum',       # total de dinero vendido
        'cantidad': 'sum',    # suma los productos vendidos
        'id_pedido': 'count'  # contar pedidos
    })
    .rename(columns={
        'total': 'total_vendido',
        'cantidad': 'cantidad_total',
        'id_pedido': 'numero_pedidos'
    })
    .sort_index()
    .head(5)
)

print(ventas_por_mes)

# 16.2 Obten la misma información pero en lugar de por mes, por semana.
print(f'\nObten la misma información pero en lugar de por mes, por semana.')

pedidos_completos_df['semana'] = pedidos_completos_df['fecha_pedido'].dt.to_period('W').astype(str)

# Agrupar por semana
ventas_por_semana = (
    pedidos_completos_df.groupby('semana')
    .agg({
        'total': 'sum',        # dinero total vendido
        'cantidad': 'sum',     # unidades totales vendidas
        'id_pedido': 'count'   # número de pedidos
    })
    .rename(columns={
        'total': 'total_vendido',
        'cantidad': 'cantidad_total',
        'id_pedido': 'numero_pedidos'
    })
    .sort_index()
    .head(5)
)

print(ventas_por_semana)




Información sobre las ventas por mes
         total_vendido  cantidad_total  numero_pedidos
mes                                                   
2023-09       16884.84            15.0               2
2023-10      331298.24           341.0              45
2023-11       77184.28            72.0              16
2023-12      112884.91           124.0              33
2024-01       74943.79            76.0              10

Obten la misma información pero en lugar de por mes, por semana.
                       total_vendido  cantidad_total  numero_pedidos
semana                                                              
2023-09-25/2023-10-01       35005.60            28.0               4
2023-10-02/2023-10-08       17026.32            15.0               3
2023-10-09/2023-10-15       52204.31            52.0               7
2023-10-16/2023-10-22       19964.22            15.0               2
2023-10-23/2023-10-29      118431.73           116.0              18


## Apartado 17: Duplicados

In [135]:
## 17.1 Contar filas duplicadas en cada uno de los datasets originales.



## Apartado 18: Valores faltantes

In [136]:
# 18.1 Obtener el porcentaje de valores faltantes en cada columna del dataset.


## Apartado 19: Valoración final

Tras completar este ejercicio, ¿qué conclusiones has obtenido acerca de los datos? ¿Consideras que sería necesario aplicar algún tipo de preprocesamiento o crees que los datos son adecuados tal como están? Apoya tu valoración con ejemplos concretos de columnas que ilustren tu análisis.

<!-- Responde aquí al apartado 19 -->
Creo que las validaciones antes de insertar en una base de datos de cualquier tipo (json, csv, parket o el que necesites) son necesarias siempre para normalizar los todos los datos y que no existan discrepancias ni categorias mal configuradas. Ya existen muchas herramientas de lenguajes tipados para poder bloquear negativos, NaN o ##ERROR## como fueron las tablas totales y cantidades, o las categorias mal nombradas ("Ro" /"Ropa) que tuve que reemplazar con un mapeo.

Muchos de estos errores puede resultar en no contar alguna columna para algún cálculo.
